## Chapter 4, P6: XGBoost API Walkthrough

Applying XGBoost requires a working knowledge of its Python interface. The library offers two primary methods for model construction:
1. Its native Python API and
2. A wrapper class compatible with the Scikit-Learn API. 

The native API will be examined first, as it r**eveals the main components of the library** and **provides the greatest flexibility.**

## The Core Data Structure: DMatrix

Unlike Scikit-Learn estimators that work directly with NumPy arrays or Pandas DataFrames, XGBoost's API uses a `DMatrix`, a memory-efficient and performance-optimised data container.

Converting your data into a `DMatrix` from its original data type (e.g. NumPy array, Pandas DataFrame) is the first step.

When creating a `DMatrix` for training, you provide both the **feature matrix** (your `X` data) and the **target vector** (your `y` data) using the `label` argument.


In [7]:
import xgboost as xgb
import numpy as np
import pandas as pd

# Generate sample training data
X_train_data = np.random.rand(100, 5)
y_train_data = np.random.rand(100)

# Create a DMatrix from a NumPy array
dtrain = xgb.DMatrix(X_train_data, label = y_train_data)

print(f"Type of the created object: {type(dtrain)}")

Type of the created object: <class 'xgboost.core.DMatrix'>


* The `dtrain` object is now ready for high-speed **training**.
* For **prediction**, you create a `DMatrix` in the same way but omit the `label` argument.

## Specifying Model Hyperparameters

In the native API, hyperparameters are not passed as arguments to a model constructor. Instead, they are defined in a Python dictionary.

Let's define a basic parameter dictionary for a regression task.

In [8]:
# Define the model's hyperparameters in a dictionary
params = {
    'objective': 'reg:squarederror',  # The loss function to be minimized
    'max_depth': 3,                   # Maximum depth of each decision tree
    'eta': 0.1,                       # Learning rate, also known as 'learning_rate'
    'eval_metric': 'rmse'             # The metric used for evaluation on a validation set
}

* `objective`: specifies the loss function to be minimised. Common loss functions include:
  * `reg:squarederror` for regression
  * `binary:logistic` for binary classification
  * `multi:softmax` for multi-class classification


## Training the Model with xgb.train

With the **training data in a `DMatrix`** and the **parameters in a dictionary**, you can train a model using the `xgb.train()` function. This function requires at least three arguments:

1. `params`: The dictionary of hyperparameters.  

2. `dtrain`: The DMatrix containing the training data.  

3. `num_boost_round`: The total number of trees to build, equivalent to `n_estimators` in Scikit-Learn.

In [10]:
# Set the number of boosting rounds
num_boost_round = 50

# Train the model
bst = xgb.train(params, dtrain, num_boost_round)

## Generating Predictions

To make predictions, you first convert your test dataset into a `DMatrix` **(without the label).**

In [14]:
# Generate sample test data
X_test_data = np.random.rand(20, 5)

# Convert the test data into a DMatrix
dtest = xgb.DMatrix(X_test_data)

# Generate predictions
predictions = bst.predict(dtest)

print("Sample predictions:")
print(predictions[:5])

Sample predictions:
[0.5833638  0.42818606 0.38423774 0.6444792  0.45481128]


* The output is a NumPy array containing the model's predictions for each sample in the test set.

## The Scikit-Learn Wrapper API

While the native API provides full control, XGBoost also includes a Scikit-Learn compatible wrapper.

This is extremely convenient if you are already familiar with Scikit-Learn's `.fit()` and `.predict()` syntax **or** if you want to **integrate XGBoost into a Scikit-Learn** `Pipeline` or **hyperparameter search tool** like `GridSearchCV`.

The primary classes are `XGBRegressor` for regression and `XGBClassifier` for classification.

In [ ]:
## Let's replicate our regression task using the XGBRegressor
from statistics import mean
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Use the same data as before
X, y = X_train_data, y_train_data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

# Instantiate the model with Scikit-Learn syntax
xgb_reg = XGBRegressor(
    objective = 'reg:squarederror',
    n_estimators = 50,
    learning_rate = 0.1,
    max_depth = 3,
    random_state = 42
)

# Fit the model using the familiar .fit() method
xgb_reg.fit(X_train, y_train)

# Make predictions using the .predict() method - NumPy array of predicted continuous values
predictions_sklearn = xgb_reg.predict(X_test)

# Evaluate the model
rmse = np.sqrt(mean_squared_error(y_test, predictions_sklearn))
print(f"RMSE with Scikit-Learn wrapper: {rmse:.4f}")

(20,)

* You don't need to manually create `DMatrix` objects; the wrapper handles the data conversion internally.
* For many applications, especially those involving **cross-validation and automated tuning**, the Scikit-Learn wrapper is the more practical choice.